# Pump It Up — Part 3 of 3: Interactive Annexes

**Competition:** [DrivenData — Pump It Up: Data Mining the Water Table](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/)

**Pipeline position:** `01 Data Preparation → 02 Model Training → [ 03 Interactive Annexes ]`

**Prerequisites:** run notebooks 01 and 02 first — this notebook reads `artifacts/feature_artifacts.pkl`, `artifacts/X_eng.parquet` and `artifacts/rf_final.joblib`. Annexes A and C additionally require an Anthropic API key in a `.env` file (`ANTHROPIC_API_KEY=sk-ant-...`).

---

## Table of Contents

**SETUP**
- Section 1 — Imports, data, model and pipeline loading

**INTERACTIVE LAYER**
- Annex A — EDA Analysis Agent with LangGraph + Claude (`http://127.0.0.1:7861`)
- Annex B — Data Explorer & Individual Predictor with Gradio (`http://127.0.0.1:7860`)
- Annex C — "Ask the Project" conversational AI assistant (`http://127.0.0.1:7862`)


---

## 1. Imports, data, model and pipeline loading


In [1]:
# Library installation for THIS notebook only.
# The interactive layer needs: gradio (Annex B), langgraph + langchain-anthropic
# (Annex A), anthropic + python-dotenv (Annexes A and C), pyarrow, joblib.
import subprocess, sys, importlib

def install(pkg, import_name=None):
    mod = import_name or pkg.replace('-', '_')
    try:
        importlib.import_module(mod)
        print(f'  OK  {pkg} (already installed)')
    except ImportError:
        print(f'  Installing {pkg}...')
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg,
                            '--quiet', '--no-warn-script-location'],
                           capture_output=True, text=True)
        print(f'  {"OK" if r.returncode == 0 else "ERROR"}  {pkg}')

install('gradio')
install('langgraph')
install('langchain-anthropic', 'langchain_anthropic')
install('anthropic')
install('python-dotenv', 'dotenv')
install('pyarrow')
install('joblib')


d:\04_Repositorios\14_Machine_Learning_3\TAREA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  OK  gradio (already installed)
  OK  langgraph (already installed)
  OK  langchain-anthropic (already installed)
  OK  anthropic (already installed)
  OK  python-dotenv (already installed)
  OK  pyarrow (already installed)
  OK  joblib (already installed)


In [2]:
# Core imports + API key from .env
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.ensemble import RandomForestClassifier
from IPython.display import display, HTML, IFrame

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
RANDOM_STATE = 42

from dotenv import load_dotenv
load_dotenv(override=True)
ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
if ANTHROPIC_API_KEY:
    print(f'OK  ANTHROPIC_API_KEY loaded ({ANTHROPIC_API_KEY[:12]}...)')
else:
    print('WARN  ANTHROPIC_API_KEY not found — Annexes A and C will not start.')
    print('      Create a .env file next to this notebook with the key.')


OK  ANTHROPIC_API_KEY loaded (sk-ant-api03...)


In [3]:
# ── IB Brand Style + shared plotting helpers ─────────────────────────────────
# Loads ib_style.py if present (same folder). Falls back to defaults otherwise.
import sys, os
import numpy as np
import matplotlib.pyplot as plt

try:
    sys.path.insert(0, os.getcwd())
    from ib_style import (
        apply_style, apply_seaborn_style, styled_fig,
        MPL, cmap_blue, cmap_orange, importance_colors,
        plot_confusion_matrix as ib_cm, style_geo_ax,
        get_gradio_theme, get_gradio_css, html_ai_assistant,
    )
    apply_style()
    apply_seaborn_style()
    IB_STYLE = True
    print('OK  IB brand style loaded and applied.')
except (ImportError, FileNotFoundError) as e:
    IB_STYLE = False
    print(f'WARN  ib_style.py not found ({e}) — default matplotlib style.')
    def styled_fig(nrows, ncols, figsize=(8, 6), title=None, subtitle=None):
        fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
        if title:
            fig.suptitle(title, fontsize=14, fontweight='bold')
        return fig, axes
    MPL = {'text':'#111111','bg':'white','bg_ax':'#fafafa','text2':'#555555',
           'functional':'#4C9BE8','needs_repair':'#E87C4C','non_functional':'#E74C3C',
           'blue':'#4C9BE8','orange':'#E87C4C','green':'#4CB87A',
           'grid':'#dddddd','teal':'#16a085','purple':'#8e44ad','red':'#E74C3C'}

C_FUNC   = MPL['functional']
C_REPAIR = MPL['needs_repair']
C_NONFUN = MPL['non_functional']
BG       = MPL['bg']
BG_AX    = MPL['bg_ax']
TEXT2    = MPL['text2']

STATUS_PALETTE = {
    'functional':              C_FUNC,
    'functional needs repair': C_REPAIR,
    'non functional':          C_NONFUN,
}


✓ IB brand style applied – matplotlib defaults updated.
✓ Seaborn IB theme applied.
OK  IB brand style loaded and applied.


In [4]:
# ── Load everything produced by notebooks 01 and 02 ───────────────────────────
import joblib
from pathlib import Path

ARTIFACTS_DIR = Path('artifacts')
required = ['feature_artifacts.pkl', 'X_eng.parquet', 'rf_final.joblib']
missing  = [f for f in required if not (ARTIFACTS_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f'Missing artifacts {missing}. Run notebooks 01 and 02 first.')

# Feature pipeline — same transformation used to build X_eng
from feature_pipeline import load_artifacts, engineer_features
load_artifacts(ARTIFACTS_DIR / 'feature_artifacts.pkl')

# Engineered matrix (column order + medians for the predictor's alignment)
X_eng = pd.read_parquet(ARTIFACTS_DIR / 'X_eng.parquet')

# Final trained model
rf_final = joblib.load(ARTIFACTS_DIR / 'rf_final.joblib')
print(f'Model loaded: {type(rf_final).__name__} '
      f'({rf_final.n_estimators} trees, {len(rf_final.classes_)} classes)')

# Raw data — needed by the EDA explorer and both AI assistants
def locate_data_folder():
    current = Path.cwd()
    for folder in [current] + list(current.parents):
        if all((folder / f).exists() for f in
               ['train_features.csv', 'train_labels.csv', 'test_features.csv']):
            return folder
        fb = folder / 'train_test_data'
        if all((fb / f).exists() for f in
               ['train_features.csv', 'train_labels.csv', 'test_features.csv']):
            return fb
    raise FileNotFoundError('Competition CSVs not found.')

DATA_DIR = locate_data_folder()
train_features = pd.read_csv(DATA_DIR / 'train_features.csv')
train_labels   = pd.read_csv(DATA_DIR / 'train_labels.csv')
df = train_features.merge(train_labels, on='id')
print(f'Raw train data loaded: {df.shape}')

# ── Tuned decision threshold + shared prediction helper ──────────────────────
# BEST_THR was computed in 02_model_training (section 6.1). The 'main model'
# for detecting 'functional needs repair' is rf_final + this threshold, not
# the raw RandomForest argmax — both annexes below use this same criterion.
import json as _json
with open(ARTIFACTS_DIR / 'model_config.json') as f:
    BEST_THR = _json.load(f)['BEST_THR']
print(f'Decision threshold loaded: BEST_THR = {BEST_THR:.2f}')

def predict_with_threshold(model, X_row, threshold=None):
    # Applies the threshold adjustment on 'functional needs repair'.
    # Returns (pred_final, proba_dict, idx_repair, threshold_used).
    if threshold is None:
        threshold = BEST_THR
    proba = model.predict_proba(X_row)[0]
    classes = list(model.classes_)
    idx_rep = classes.index('functional needs repair')
    if proba[idx_rep] >= threshold:
        pred_final = 'functional needs repair'
    else:
        p_rest = proba.copy()
        p_rest[idx_rep] = 0
        pred_final = classes[int(np.argmax(p_rest))]
    return pred_final, dict(zip(classes, proba)), idx_rep, threshold


Model loaded: RandomForestClassifier (500 trees, 3 classes)
Raw train data loaded: (59400, 41)
Decision threshold loaded: BEST_THR = 0.05


---

### Annex A — EDA Analysis Agent with LangGraph and LangChain

LangGraph allows building agents as state graphs: each node is a function that receives the current state and returns an updated state. Here it is used to orchestrate an agent that systematically goes through the dataset, analyses each EDA aspect (missing values, distributions, correlations, quality) and generates automatic conclusions using an LLM.

The graph has four sequential nodes:
`analyse_structure` -> `analyse_quality` -> `analyse_distributions` -> `generate_summary`

Each node enriches the state with its findings; the final node synthesises everything into an executive report.


In [5]:
# EDA Agent with LangGraph + HTML interface at http://127.0.0.1:7861
# The graph generates the EDA report in 4 nodes then starts an HTTP server
# with interactive chat to answer questions about the results.

# Safety check: ensure df has status_group (in case df was overwritten
# by feature engineering steps earlier in the notebook)
import pandas as pd
if 'status_group' not in df.columns:
    if 'train_features' in globals() and 'train_labels' in globals():
        df = train_features.merge(train_labels, on='id')
    else:
        from pathlib import Path
        _current = Path.cwd()
        _data_dir = None
        for _folder in [_current] + list(_current.parents):
            if (_folder / 'train_features.csv').exists() and (_folder / 'train_labels.csv').exists():
                _data_dir = _folder
                break
            _fallback = _folder / 'Mejora_NeedsRepair'
            if (_fallback / 'train_features.csv').exists() and (_fallback / 'train_labels.csv').exists():
                _data_dir = _fallback
                break
        if _data_dir is None:
            raise FileNotFoundError('train_features.csv / train_labels.csv not found')
        _tf = pd.read_csv(_data_dir / 'train_features.csv')
        _tl = pd.read_csv(_data_dir / 'train_labels.csv')
        df = _tf.merge(_tl, on='id')

import threading
from http.server import HTTPServer, BaseHTTPRequestHandler
from urllib.parse import parse_qs, urlparse

try:
    from langgraph.graph import StateGraph, END
    from langchain_anthropic import ChatAnthropic
    from typing import TypedDict

    # ANTHROPIC_API_KEY is loaded from .env in cell 1 (imports)

    if not ANTHROPIC_API_KEY:
        print("API key no configurada.")
        LANGGRAPH_OK = False
    else:

        class EDAState(TypedDict):
            dataset_info:   str
            quality_report: str
            dist_report:    str
            final_summary:  str

        llm_eda = ChatAnthropic(
            model="claude-sonnet-4-5",
            api_key=ANTHROPIC_API_KEY,
            max_tokens=700
        )

        # ── Node 1: basic statistics (no LLM) ──────────────────────────
        def analizar_estructura(state):
            vc = df["status_group"].value_counts()
            info = (
                f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}\n"
                f"Classes:\n" +
                "\n".join([f"  {k}: {v:,} ({v/len(df)*100:.1f}%)"
                            for k, v in vc.items()]) +
                f"\nNumericas: {len(df.select_dtypes(include='number').columns)} | "
                f"Categorical: {len(df.select_dtypes(include='object').columns)}"
            )
            state["dataset_info"] = info
            return state

        # ── Node 2: quality analysis with LLM ─────────────────────────────
        def analizar_calidad(state):
            miss = df.isnull().sum()
            miss_str = "\n".join([
                f"  {c}: {n} ({n/len(df)*100:.1f}%)"
                for c, n in miss[miss > 0].items()
            ])
            zero_str = "\n".join([
                f"  {c}: {(df[c]==0).sum():,} ceros ({(df[c]==0).mean()*100:.1f}%)"
                for c in ["longitude","latitude","construction_year","population","amount_tsh"]
                if (df[c]==0).sum() > 0
            ])
            prompt = (
                "In 3 concise sentences describe the quality issues of this "
                "Tanzania water pumps dataset for an ML model.\n\n"
                f"Missing values:\n{miss_str}\n\n"
                f"Zeros as missing data:\n{zero_str}\n\n"
                "Reply in English without markdown formatting."
            )
            state["quality_report"] = llm_eda.invoke(prompt).content
            return state

        # ── Node 3: predictive patterns with LLM ────────────────────────────
        def analizar_distribuciones(state):
            qty_nf = (
                df.groupby("quantity")["status_group"]
                .apply(lambda x: (x=="non functional").mean()*100).round(1)
            )
            pay_nf = (
                df.groupby("payment")["status_group"]
                .apply(lambda x: (x=="non functional").mean()*100).round(1)
            )
            yr_f  = df[df.status_group=="functional"]["construction_year"].replace(0,None).dropna().mean()
            yr_nf = df[df.status_group=="non functional"]["construction_year"].replace(0,None).dropna().mean()
            stats = (
                f"Tasa no-funcional por quantity:\n{qty_nf.to_string()}\n\n"
                f"Tasa no-funcional por payment:\n{pay_nf.to_string()}\n\n"
                f"Anio medio construccion: funcional={yr_f:.0f}, no-funcional={yr_nf:.0f}"
            )
            prompt = (
                "Identifica los 3 factores mas importantes que predicen el fallo "
                "de una bomba de agua en Tanzania. "
                "Responde en espanol con bullet points cortos.\n\n" + stats
            )
            state["dist_report"] = llm_eda.invoke(prompt).content
            return state

        # ── Node 4: executive report with LLM ────────────────────────────────
        def generar_resumen(state):
            prompt = (
                "Write a 4-5 sentence executive paragraph for non-technical stakeholders "
                "summarising the EDA findings of this Tanzania water pumps dataset "
                "and the implications for the ML model. Reply in English.\n\n"
                f"Estructura: {state['dataset_info']}\n\n"
                f"Calidad: {state['quality_report']}\n\n"
                f"Patrones: {state['dist_report']}"
            )
            state["final_summary"] = llm_eda.invoke(prompt).content
            return state

        # ── Compile and run the graph ─────────────────────────────────────
        graph_eda = StateGraph(EDAState)
        for name, fn in [
            ("analizar_estructura",     analizar_estructura),
            ("analizar_calidad",        analizar_calidad),
            ("analizar_distribuciones", analizar_distribuciones),
            ("generar_resumen",         generar_resumen),
        ]:
            graph_eda.add_node(name, fn)

        graph_eda.set_entry_point("analizar_estructura")
        graph_eda.add_edge("analizar_estructura",     "analizar_calidad")
        graph_eda.add_edge("analizar_calidad",        "analizar_distribuciones")
        graph_eda.add_edge("analizar_distribuciones", "generar_resumen")
        graph_eda.add_edge("generar_resumen",         END)
        app_eda = graph_eda.compile()

        print("Running LangGraph EDA agent...")
        eda_result = app_eda.invoke(
            EDAState(dataset_info="", quality_report="",
                     dist_report="", final_summary="")
        )
        print("EDA report generated. Starting HTML interface...")

        # ── Chat context and memory ──────────────────────────────────────
        EDA_CTX = "\n\n".join([
            f"ESTRUCTURA:\n{eda_result['dataset_info']}",
            f"CALIDAD:\n{eda_result['quality_report']}",
            f"PATRONES:\n{eda_result['dist_report']}",
            f"RESUMEN:\n{eda_result['final_summary']}",
        ])
        chat_eda = []

        def ask_eda(question):
            hist = "\n".join([
                f"{m['role']}: {m['content'][:200]}"
                for m in chat_eda[-6:]
            ])
            prompt = (
                f"Eres analista de datos experto. Informe EDA:\n\n{EDA_CTX}\n\n"
                f"Historial:\n{hist}\n\nPregunta: {question}\n\n"
                "Responde en espanol de forma concisa y tecnica."
            )
            resp = llm_eda.invoke(prompt).content
            chat_eda.append({"role": "user",      "content": question})
            chat_eda.append({"role": "assistant", "content": resp})
            return resp

        # ── Interface HTML ──────────────────────────────────────────────
        def build_eda_html():
            sections = [
                ("Dataset Structure",  eda_result["dataset_info"]),
                ("Quality Analysis",     eda_result["quality_report"]),
                ("Predictive Patterns",    eda_result["dist_report"]),
                ("Executive Report",       eda_result["final_summary"]),
            ]
            report_html = ""
            for title, content in sections:
                safe = (
                    content
                    .replace("&", "&amp;")
                    .replace("<", "&lt;")
                    .replace(">", "&gt;")
                    .replace("\n", "<br>")
                )
                report_html += (
                    f'<div class="section">'
                    f'<h3>{title}</h3>'
                    f'<p>{safe}</p>'
                    f'</div>'
                )

            msgs_html = ""
            for m in chat_eda:
                cls = "user-msg" if m["role"] == "user" else "bot-msg"
                lbl = "Tu" if m["role"] == "user" else "Agente"
                safe = (
                    m["content"]
                    .replace("&", "&amp;")
                    .replace("<", "&lt;")
                    .replace(">", "&gt;")
                    .replace("\n", "<br>")
                )
                msgs_html += (
                    f'<div class="msg {cls}">'
                    f'<strong>{lbl}:</strong> {safe}'
                    f'</div>'
                )

            empty = "" if msgs_html else '<p class="empty">Escribe una pregunta sobre el EDA</p>'

            return (
                "<!DOCTYPE html><html lang='es'><head>"
                "<meta charset='UTF-8'>"
                "<meta name='viewport' content='width=device-width,initial-scale=1'>"
                "<title>Pump It Up EDA Agent</title>"
                "<style>"
                "*{box-sizing:border-box;margin:0;padding:0}"
                "body{font-family:'Segoe UI',Arial,sans-serif;background:#0f1117;"
                "     color:#e0e0e0;min-height:100vh}"
                ".header{background:#1a1d27;padding:24px 32px;"
                "        border-bottom:3px solid #4C9BE8}"
                ".header h1{color:#fff;font-size:1.6em;margin-bottom:4px}"
                ".badge{background:#4C9BE8;color:#fff;font-size:.72em;"
                "       padding:2px 9px;border-radius:10px;"
                "       vertical-align:middle;margin-left:8px}"
                ".header p{color:#aaa;font-size:.88em}"
                ".main{max-width:960px;margin:0 auto;padding:24px 20px}"
                ".section{background:#1a1d27;border-left:4px solid #4C9BE8;"
                "         border-radius:6px;padding:16px 20px;margin-bottom:14px}"
                ".section h3{color:#4C9BE8;font-size:.95em;text-transform:uppercase;"
                "            letter-spacing:.06em;margin-bottom:9px}"
                ".section p{line-height:1.75;color:#ccc;font-size:.93em}"
                ".chat-area{background:#1a1d27;border-radius:8px;"
                "           overflow:hidden;margin-top:24px}"
                ".chat-title{background:#162030;color:#4C9BE8;"
                "            padding:13px 20px;font-weight:600;font-size:.95em}"
                ".sugs{padding:12px 20px;background:#13151f;"
                "      display:flex;flex-wrap:wrap;gap:8px;border-bottom:1px solid #222}"
                ".sug{padding:5px 12px;background:#1a2e1a;"
                "     border:1px solid #4CB87A44;border-radius:14px;"
                "     font-size:.8em;cursor:pointer;color:#4CB87A}"
                ".sug:hover{background:#1e3a1e}"
                ".msgs{min-height:100px;max-height:380px;overflow-y:auto;"
                "      padding:16px 20px;display:flex;flex-direction:column;gap:10px}"
                ".empty{color:#555;font-size:.88em;text-align:center;padding:20px 0}"
                ".msg{padding:10px 14px;border-radius:6px;"
                "     font-size:.92em;line-height:1.65;max-width:94%}"
                ".msg strong{display:block;font-size:.8em;opacity:.65;margin-bottom:3px}"
                ".user-msg{background:#1e3a5f;border-left:3px solid #4C9BE8;"
                "          align-self:flex-end}"
                ".bot-msg{background:#1a2e1a;border-left:3px solid #4CB87A;"
                "         align-self:flex-start}"
                ".input-row{display:flex;gap:9px;padding:14px 20px;"
                "           background:#13151f;border-top:1px solid #222}"
                "input{flex:1;padding:11px 14px;background:#1a1d27;"
                "      border:1px solid #333;border-radius:6px;"
                "      color:#e0e0e0;font-size:.92em;outline:none}"
                "input:focus{border-color:#4C9BE8}"
                "button{padding:11px 20px;background:#4C9BE8;color:#fff;"
                "       border:none;border-radius:6px;cursor:pointer;"
                "       font-size:.92em;font-weight:600}"
                "button:hover{background:#3a89d4}"
                "</style></head><body>"
                "<div class='header'>"
                "<h1>Pump It Up EDA Agent "
                "<span class='badge'>LangGraph + Claude</span></h1>"
                "<p>Automated Exploratory Analysis | Tanzania Water Pumps</p>"
                "</div>"
                "<div class='main'>"
                f"{report_html}"
                "<div class='chat-area'>"
                "<div class='chat-title'>Chat with the EDA Agent</div>"
                "<div class='sugs'>"
                "<span class='sug' onclick='fillQ(this)'>"
                "Which variable is most predictive of failure?</span>"
                "<span class='sug' onclick='fillQ(this)'>"
                "Why are there so many zeros in amount_tsh?</span>"
                "<span class='sug' onclick='fillQ(this)'>"
                "How does class imbalance affect the model?</span>"
                "<span class='sug' onclick='fillQ(this)'>"
                "Which regions have the most non-functional pumps?</span>"
                "</div>"
                f"<div class='msgs' id='msgs'>{msgs_html or empty}</div>"
                "<div class='input-row'>"
                "<input type='text' id='q' "
                "       placeholder='Ask about the EDA...'"
                "       onkeydown='if(event.key==\"Enter\")send()'>"
                "<button onclick='send()'>Enviar</button>"
                "</div></div></div>"
                "<script>"
                "function send(){"
                "  var q=document.getElementById('q').value.trim();"
                "  if(!q)return;"
                "  document.getElementById('q').value='';"
                "  fetch('/chat_eda?q='+encodeURIComponent(q))"
                "    .then(r=>r.json()).then(()=>location.reload());"
                "}"
                "function fillQ(el){"
                "  document.getElementById('q').value=el.textContent;"
                "}"
                "window.onload=function(){"
                "  var m=document.getElementById('msgs');"
                "  if(m)m.scrollTop=m.scrollHeight;"
                "};"
                "</script></body></html>"
            )

        # ── HTTP Server ────────────────────────────────────────────────────
        class EDAHandler(BaseHTTPRequestHandler):
            def log_message(self, f, *a): pass
            def do_GET(self):
                p = urlparse(self.path)
                if p.path == "/chat_eda":
                    q = parse_qs(p.query).get("q", [""])[0]
                    if q:
                        ask_eda(q)
                    self.send_response(200)
                    self.send_header("Content-Type", "application/json")
                    self.end_headers()
                    self.wfile.write(b'{"ok":true}')
                else:
                    html = build_eda_html().encode("utf-8")
                    self.send_response(200)
                    self.send_header("Content-Type", "text/html; charset=utf-8")
                    self.send_header("Content-Length", str(len(html)))
                    self.end_headers()
                    self.wfile.write(html)

        PORT_EDA = 7861
        httpd_eda = HTTPServer(("127.0.0.1", PORT_EDA), EDAHandler)
        t_eda = threading.Thread(target=httpd_eda.serve_forever, daemon=True)
        t_eda.start()

        print(f"EDA interface available at: http://127.0.0.1:{PORT_EDA}")
        print("Open it in your browser. To stop it: httpd_eda.shutdown()")
        LANGGRAPH_OK = True

except ImportError as e:
    print(f"Library not available: {e}")
    print("pip install langgraph langchain-anthropic")
    LANGGRAPH_OK = False
except Exception as e:
    print(f"Error: {e}")
    LANGGRAPH_OK = False


Running LangGraph EDA agent...
EDA report generated. Starting HTML interface...
EDA interface available at: http://127.0.0.1:7861
Open it in your browser. To stop it: httpd_eda.shutdown()


---

### Annex B — Interactive Interface with Gradio and Model Demo

Gradio allows creating interactive web interfaces directly from the notebook. Two interfaces are built:

**Interactive EDA panel:** allows selecting any categorical variable from the dataset and instantly seeing the pump status distribution for that variable. Useful for quickly exploring variables not analysed in detail.

**Model demo:** allows entering the values for a specific pump and getting the model prediction with the probability of each class. Useful for demonstrating the model to a non-technical audience or for inspecting individual cases.


In [7]:
# Annex B — Interactive Gradio Interface with IB Brand Style
import pandas as pd

try:
    import gradio as gr
    import io
    from PIL import Image as PILImage

    cat_options = ['quantity', 'waterpoint_type', 'extraction_type_class',
                   'payment', 'water_quality', 'basin', 'source',
                   'management', 'scheme_management', 'permit', 'public_meeting']

    PASTEL = {
        'functional':              '#7db7f0',
        'functional needs repair': '#fdb36b',
        'non functional':          '#f48d8d',
    }

    def plot_cat_vs_target(variable):
        ct = pd.crosstab(df[variable], df['status_group'], normalize='index') * 100
        cols_order = [c for c in ['functional', 'functional needs repair', 'non functional']
                      if c in ct.columns]
        ct = ct[cols_order].sort_values('functional', ascending=True)
        h  = max(4, len(ct) * 0.52)
        if IB_STYLE:
            fig, ax = styled_fig(1, 1, figsize=(10, h))
        else:
            fig, ax = plt.subplots(figsize=(10, h))
        colors_plot = [PASTEL.get(c, '#aaa') for c in cols_order]
        ct.plot(kind='barh', ax=ax, color=colors_plot, edgecolor='none', width=0.65)
        ax.set_title(f'Pump status by {variable}', fontsize=11, fontweight='bold')
        ax.set_xlabel('Percentage (%)', fontsize=9)
        ax.legend(loc='lower right', fontsize=8, framealpha=0)
        ax.grid(False)
        for sp in ax.spines.values(): sp.set_visible(False)
        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=130, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        return PILImage.open(buf)

    def predecir_estado(quantity, payment, waterpoint_type, extraction_type_class,
                         water_quality, basin, construction_year, gps_height,
                         population, longitude, latitude):
        try:
            row = {
                'id': 0, 'amount_tsh': 0, 'funder': 'Government',
                'gps_height': int(gps_height), 'installer': 'Government',
                'longitude': float(longitude), 'latitude': float(latitude),
                'wpt_name': 'none', 'num_private': 0, 'basin': basin,
                'subvillage': 'unknown', 'region': 'Dodoma', 'region_code': 1,
                'district_code': 1, 'lga': 'Dodoma', 'ward': 'Dodoma',
                'population': int(population), 'public_meeting': True,
                'recorded_by': 'GeoData Consultants Ltd', 'scheme_management': 'VWC',
                'scheme_name': 'K', 'permit': True,
                'construction_year': int(construction_year),
                'extraction_type': extraction_type_class,
                'extraction_type_group': extraction_type_class,
                'extraction_type_class': extraction_type_class,
                'management': 'vwc', 'management_group': 'user-group',
                'payment': payment, 'payment_type': payment,
                'water_quality': water_quality, 'quality_group': 'good',
                'quantity': quantity, 'quantity_group': quantity,
                'source': 'spring', 'source_type': 'spring',
                'source_class': 'groundwater', 'waterpoint_type': waterpoint_type,
                'waterpoint_type_group': waterpoint_type, 'date_recorded': '2013-03-01',
            }
            row_df  = pd.DataFrame([row])
            row_eng = engineer_features(row_df)
            for col in X_eng.columns:
                if col not in row_eng.columns:
                    row_eng[col] = X_eng[col].median()
            cols_extra = [c for c in row_eng.columns if c not in X_eng.columns]
            if cols_extra:
                row_eng.drop(columns=cols_extra, inplace=True)
            row_eng = row_eng[X_eng.columns]
            # Modelo principal: rf_final + threshold ajustado para 'needs repair'
            pred, proba_dict, idx_rep, thr_used = predict_with_threshold(rf_final, row_eng)
            classes = list(proba_dict.keys())
            proba   = np.array(list(proba_dict.values()))
            raw_pred = rf_final.predict(row_eng)[0]
            raw_proba_repair = proba_dict['functional needs repair']
            color_map = {
                'functional':              '#3b82f6',
                'functional needs repair': '#f97316',
                'non functional':          '#ef4444',
            }
            pred_color = color_map.get(pred, '#888888')
            bars_html = ''
            for cls, p in sorted(zip(classes, proba), key=lambda x: -x[1]):
                c     = color_map.get(cls, '#888')
                width = max(4, int(p * 220))
                bars_html += (
                    f'<div style="margin:10px 0;display:flex;align-items:center;gap:12px">'
                    f'<span style="width:220px;font-size:.88em;color:#94a3b8">{cls}</span>'
                    f'<span style="display:inline-block;width:{width}px;height:14px;'
                    f'background:{c};border-radius:7px;opacity:.9"></span>'
                    f'<span style="font-size:.9em;color:#e2e8f0;font-weight:600">{p*100:.1f}%</span>'
                    f'</div>'
                )
            # Bloque de detalle técnico: threshold aplicado y comparación con
            # la predicción cruda del RandomForest (sin ajuste de threshold)
            adjustment_note = ''
            if raw_pred != pred:
                adjustment_note = (
                    f'<div style="margin-top:14px;padding:10px 12px;border-radius:8px;'
                    f'background:#1a2332;border:1px solid #334155;font-size:.78em;'
                    f'color:#94a3b8;line-height:1.5">'
                    f'&#9888; Threshold adjustment changed the prediction: '
                    f'raw RandomForest prediction was '
                    f'<strong style="color:#e2e8f0">{raw_pred}</strong>.'
                    f'</div>'
                )
            tech_detail = (
                f'<div style="margin-top:14px;padding:10px 12px;border-radius:8px;'
                f'background:#0f172a;border:1px solid #1e293b;font-size:.75em;'
                f'color:#64748b;font-family:monospace;line-height:1.6">'
                f'P(needs repair) raw = {raw_proba_repair:.3f}  |  '
                f'threshold = {thr_used:.2f}  |  '
                f'decision: {"needs repair (>= threshold)" if raw_proba_repair >= thr_used else "below threshold -> next best class"}'
                f'</div>'
            )
            return (
                f'<div style="padding:24px;border-radius:12px;'
                f'background:linear-gradient(135deg,#0d1628,#111827);'
                f'border:2px solid {pred_color};font-family:Inter,sans-serif;">'
                f'<div style="font-size:1.4em;font-weight:700;color:{pred_color};'
                f'margin-bottom:16px;letter-spacing:.02em">'
                f'&#9679; Prediction: {pred.upper()}</div>'
                f'<div style="margin-bottom:10px;font-size:.78em;color:#475569;'
                f'text-transform:uppercase;letter-spacing:.1em">Probability by class</div>'
                f'{bars_html}'
                f'{tech_detail}'
                f'{adjustment_note}'
                f'</div>'
            )
        except Exception as e:
            import traceback
            tb = traceback.format_exc().replace('<','&lt;').replace('>','&gt;')
            return (
                f'<div style="padding:16px;border-radius:8px;background:#1a0a0a;'
                f'border:2px solid #ef4444;font-family:monospace;font-size:.82em;color:#ef4444">'
                f'<strong>Error:</strong><br><pre>{tb}</pre></div>'
            )

    if IB_STYLE:
        theme = gr.themes.Base(**get_gradio_theme())
        css   = get_gradio_css()
    else:
        theme = gr.themes.Soft()
        css   = ''

    with gr.Blocks(title='Pump It Up — EDA and Prediction', theme=theme, css=css) as demo:
        gr.Markdown('# Pump It Up — Data Explorer and Prediction')
        gr.Markdown('Tanzania Water Pumps — DrivenData Competition')
        with gr.Tab('EDA Explorer'):
            gr.Markdown('### Select a variable to see its relationship with pump status')
            var_dropdown = gr.Dropdown(choices=cat_options, value='quantity',
                                        label='Categorical variable')
            eda_plot = gr.Image(type='pil', label='Distribution by status')
            var_dropdown.change(fn=plot_cat_vs_target, inputs=var_dropdown, outputs=eda_plot)
            demo.load(fn=lambda: plot_cat_vs_target('quantity'), outputs=eda_plot)
        with gr.Tab('Individual prediction'):
            gr.Markdown('### Enter pump data to get the model prediction')
            with gr.Row():
                with gr.Column():
                    q_input  = gr.Dropdown(
                        ['enough', 'seasonal', 'insufficient', 'dry', 'unknown'],
                        value='enough', label='Water quantity')
                    p_input  = gr.Dropdown(
                        ['never pay', 'pay annually', 'pay monthly',
                         'pay per bucket', 'pay when scheme fails', 'other'],
                        value='pay annually', label='Payment system')
                    wt_input = gr.Dropdown(
                        ['communal standpipe', 'hand pump', 'improved spring',
                         'cattle trough', 'dam', 'other'],
                        value='communal standpipe', label='Waterpoint type')
                    et_input = gr.Dropdown(
                        ['gravity', 'handpump', 'submersible', 'motorpump',
                         'rope pump', 'wind-powered', 'other'],
                        value='gravity', label='Extraction class')
                    wq_input = gr.Dropdown(
                        ['soft', 'salty', 'milky', 'coloured', 'fluoride', 'unknown'],
                        value='soft', label='Water quality')
                    ba_input = gr.Dropdown(
                        ['Lake Victoria', 'Pangani', 'Rufiji', 'Internal',
                         'Lake Tanganyika', 'Wami / Ruvu', 'Lake Nyasa',
                         'Lake Rukwa', 'Ruvuma / Southern Coast'],
                        value='Rufiji', label='Hydrographic basin')
                with gr.Column():
                    cy_input  = gr.Slider(1960, 2013, value=2000, step=1,   label='Construction year')
                    gps_input = gr.Slider(-90, 2770,  value=1200, step=10,  label='GPS altitude (m)')
                    pop_input = gr.Slider(0, 10000,   value=300,  step=50,  label='Nearby population')
                    lon_input = gr.Slider(29, 41,     value=35,   step=0.1, label='Longitude')
                    lat_input = gr.Slider(-12, -0.5,  value=-6,   step=0.1, label='Latitude')
                    pred_btn  = gr.Button('Predict status', variant='primary')
            pred_output = gr.HTML(label='Prediction result')
            pred_btn.click(
                fn=predecir_estado,
                inputs=[q_input, p_input, wt_input, et_input, wq_input, ba_input,
                        cy_input, gps_input, pop_input, lon_input, lat_input],
                outputs=pred_output
            )
    demo.launch(share=False)

except ImportError:
    print('Gradio is not installed. Run: pip install gradio')


ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 7861): [winerror 10048] solo se permite un uso de cada dirección de socket (protocolo/dirección de red/puerto)


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


AttributeError: 'str' object has no attribute 'name'

---

## Annex C — Ask the Project

There is an assistant here that knows everything about the analysis performed: the data studied, the problems encountered, the models tested and the results obtained.

You can ask it anything about the project in plain language, no technical knowledge required. Available at `http://127.0.0.1:7862`.


In [ ]:
        f'  - PRODUCTION MODEL: Optimised RF (rf_final) with adjusted decision '
        f'threshold ({BEST_THR:.2f} instead of default 0.33) for "functional needs repair".',
        f'    This is the model used by the prediction interfaces (Gradio, this assistant).',
        f'    It trades a small amount of overall accuracy for a much higher recall '
        f'on the operationally critical "needs repair" class '
        f'(repair recall ~65% vs default ~41%).',

---

**Part 3 complete.** The three interfaces run as local servers:

| Annex | URL | Requires API key |
|---|---|---|
| A — LangGraph EDA Agent | `http://127.0.0.1:7861` | Yes |
| B — Data Explorer & Predictor | `http://127.0.0.1:7860` | No |
| C — "Ask the Project" | `http://127.0.0.1:7862` | Yes |

They stay alive while the kernel is running; restarting the kernel stops all three.